In [1]:
import requests
import json
import threading
import time
from websocket import WebSocketApp
from datetime import datetime, timezone
import pandas as pd

# ----------------------------
# 1️⃣ Получение топ-рынков
# ----------------------------
url = 'https://gamma-api.polymarket.com/markets?active=true&limit=10&order=volume24hr&ascending=false'
markets = requests.get(url).json()

def test_market(market, timeout=5):
    """Проверка живости рынка: количество book-событий за timeout секунд"""
    events_count = 0
    token_ids = market.get("clobTokenIds", [])[:3]

    def on_open(ws):
        sub = {"type": "market", "assets_ids": token_ids, "custom_feature_enabled": True}
        ws.send(json.dumps(sub))

    def on_message(ws, message):
        nonlocal events_count
        if message.strip() == "PONG":
            return
        try:
            data = json.loads(message)
            if isinstance(data, list):
                for item in data:
                    if item.get("event_type") == "book":
                        events_count += 1
            else:
                if data.get("event_type") == "book":
                    events_count += 1
        except:
            pass

    ws = WebSocketApp(
        "wss://ws-subscriptions-clob.polymarket.com/ws/market",
        on_open=on_open,
        on_message=on_message
    )

    t = threading.Thread(target=ws.run_forever, daemon=True)
    t.start()
    time.sleep(timeout)
    ws.close()
    return events_count

# Проверяем первые 5 рынков
active_markets = []
for m in markets[:5]:
    cnt = test_market(m, timeout=5)
    print(f"{m['question'][:60]}... ⇒ book events in 5s: {cnt}")
    if cnt > 0:
        active_markets.append((m, cnt))

if not active_markets:
    print("❌ Нет активных рынков для арбитража")
    exit()

# Выбираем рынок с максимальным количеством событий
best_market = max(active_markets, key=lambda x: x[1])[0]
token_ids = best_market["clobTokenIds"][:3]
print(f"🚀 Выбран активный рынок: {best_market['question']} (Volume: ${best_market['volume24hr']:,})")

# ----------------------------
# 2️⃣ Симулятор negrisk-арбитража
# ----------------------------
class NegriskArbSimulator:
    FEE = 0.001

    def __init__(self, asset_ids):
        self.asset_ids = asset_ids
        self.url = "wss://ws-subscriptions-clob.polymarket.com/ws/market"

        self.prices = {aid: 0.0 for aid in asset_ids}
        self.best_bids = {aid: 0.0 for aid in asset_ids}

        self.position_open = False
        self.entry_time = None
        self.entry_cost = None
        self.entry_prices = None

        self.trades = []
        self.lock = threading.Lock()

    def heartbeat(self, ws):
        while True:
            time.sleep(20)
            try:
                ws.send("PING")
            except:
                break

    def on_open(self, ws):
        print("✅ WS подключен")
        sub = {"type": "market", "assets_ids": self.asset_ids, "custom_feature_enabled": True}
        ws.send(json.dumps(sub))
        print("📤 Подписка отправлена")
        threading.Thread(target=self.heartbeat, args=(ws,), daemon=True).start()

    def on_message(self, ws, message):
        if message.strip() == "PONG":
            print("📩 got PONG")
            return
        try:
            data = json.loads(message)
        except:
            return

        if isinstance(data, list):
            for item in data:
                self.process_event(item)
        else:
            self.process_event(data)

    def process_event(self, data):
        if data.get("event_type") != "book":
            return

        asset_id = data.get("asset_id")
        asks = data.get("asks", [])
        bids = data.get("bids", [])

        if not asks or not bids:
            return

        best_ask = float(asks[0][0])
        best_bid = float(bids[0][0])

        self.prices[asset_id] = best_ask
        self.best_bids[asset_id] = best_bid

        # если не все цены есть — пропускаем
        if 0 in self.prices.values():
            return

        total_cost = sum(self.prices.values())
        now = datetime.now(timezone.utc)

        # ENTRY: суммарная стоимость < $1
        if (not self.position_open) and (0 < total_cost < 1):
            self.position_open = True
            self.entry_time = now
            self.entry_cost = total_cost
            self.entry_prices = self.prices.copy()

        # EXIT: возможность исчезла
        elif self.position_open and total_cost >= 1:
            self.close_position(now)

    def close_position(self, now):
        duration = (now - self.entry_time).total_seconds()
        pnl = (1 - self.entry_cost) * (1 - self.FEE)
        self.trades.append({
            "entry_time": self.entry_time,
            "exit_time": now,
            "duration_sec": duration,
            "entry_cost": self.entry_cost,
            "pnl": pnl
        })
        self.position_open = False
        self.entry_time = None
        self.entry_cost = None
        self.entry_prices = None

    def on_close(self, ws, *args):
        print("🔌 WS закрыт")
        self.save_results()

    def run(self):
        while True:
            try:
                ws = WebSocketApp(
                    self.url,
                    on_open=self.on_open,
                    on_message=self.on_message,
                    on_close=self.on_close
                )
                ws.run_forever()
            except Exception as e:
                print("❌ WS ошибка:", e)
            print("♻️ Переподключение через 5 сек...")
            time.sleep(5)

    def save_results(self):
        if not self.trades:
            print("Нет сделок для сохранения")
            return

        df = pd.DataFrame(self.trades)
        df.to_csv("trades.csv", index=False)

        stats = {
            "total_trades": len(df),
            "total_pnl": df["pnl"].sum(),
            "avg_pnl": df["pnl"].mean(),
            "avg_duration_sec": df["duration_sec"].mean(),
            "max_duration_sec": df["duration_sec"].max()
        }
        pd.DataFrame([stats]).to_csv("stats.csv", index=False)

        print("\n📊 РЕЗУЛЬТАТЫ:")
        for k, v in stats.items():
            print(f"{k}: {v}")

# ----------------------------
# 3️⃣ Запуск
# ----------------------------
sim = NegriskArbSimulator(token_ids)
try:
    sim.run()
except KeyboardInterrupt:
    sim.save_results()

US forces enter Iran by April 30?... ⇒ book events in 5s: 0
US x Iran ceasefire by April 7?... ⇒ book events in 5s: 0
Will Paris Saint-Germain FC win on 2026-04-08?... ⇒ book events in 5s: 0
Iran x Israel/US conflict ends by April 7?... ⇒ book events in 5s: 0
Will Liverpool FC win on 2026-04-08?... ⇒ book events in 5s: 0
❌ Нет активных рынков для арбитража


ValueError: max() iterable argument is empty

In [2]:
import requests
import json
import threading
import time
from websocket import WebSocketApp
from datetime import datetime
import pandas as pd

# ----------------------------
# 1️⃣ Выбор ликвидного negRisk рынка
# ----------------------------
url = 'https://gamma-api.polymarket.com/markets?active=true&limit=50&order=volume24hr&ascending=false'
markets = requests.get(url).json()

target_market = None
token_ids = []

for m in markets:
    tokens = m.get('clobTokenIds', [])
    if tokens and m.get('negRisk') and m.get('acceptingOrders'):
        print(f"✅ НАШЁЛ: {m['question'][:60]}...")
        print(f"💰 Volume: ${m['volume24hr']:,}")
        token_ids = tokens[:3]  # берем только 3 токена для стабильного WS
        target_market = m
        break

if not target_market:
    print("❌ Нет подходящих рынков")
    exit()
print(f"🚀 Подписка на {len(token_ids)} токенов рынка")

# ----------------------------
# 2️⃣ WS + negrisk арбитраж + логирование сигналов
# ----------------------------
class NegriskArbWS:
    MAX_RECONNECTS = 5

    def __init__(self, asset_ids):
        self.asset_ids = asset_ids
        self.url = "wss://ws-subscriptions-clob.polymarket.com/ws/market"
        self.prices = {aid: 0.0 for aid in asset_ids}
        self.ws = None
        self.reconnect_count = 0

        self.events = []
        self.signals = []
        self.current_signal_start = None
        self.current_signal_prices = {}

    def on_open(self, ws):
        print("✅ Подключено к Polymarket WS")
        sub = {"assets_ids": json.dumps(self.asset_ids), "type": "market", "custom_feature_enabled": True}
        ws.send(json.dumps(sub))
        print(f"📤 Отправлена подписка на {len(self.asset_ids)} токенов")
        threading.Thread(target=self.heartbeat, args=(ws,), daemon=True).start()

    def on_message(self, ws, message):
        if message.strip() == "PONG": 
            return
        try:
            data = json.loads(message)
            if isinstance(data, list):
                for item in data: 
                    self.process_event(item)
            else:
                self.process_event(data)
        except:
            pass

    def process_event(self, data):
        event = data.get("event_type")
        asset_id = data.get("asset_id", "unknown")
        now = datetime.utcnow()

        if event == "book":
            asks = data.get("asks", [])
            best_ask = float(asks[0][0]) if asks else 0
            self.prices[asset_id] = best_ask
            total_cost = sum(self.prices.values())
            signal_active = 0 < total_cost < 1.0

            if signal_active:
                if self.current_signal_start is None:
                    self.current_signal_start = now
                self.current_signal_prices = dict(self.prices)
            else:
                self.close_current_signal(now)

            self.events.append({
                "timestamp": now,
                "event_type": event,
                "asset_id": asset_id,
                "best_ask": best_ask,
                "total_cost": total_cost,
                "signal_active": signal_active
            })

        elif event == "last_trade_price":
            price = float(data.get('price', 0))
            size = float(data.get('size', 0))
            side = data.get('side', '?')
            self.events.append({
                "timestamp": now,
                "event_type": event,
                "asset_id": asset_id,
                "price": price,
                "size": size,
                "side": side
            })

    def close_current_signal(self, now):
        if self.current_signal_start:
            duration = (now - self.current_signal_start).total_seconds()
            self.signals.append({
                "start_time": self.current_signal_start,
                "end_time": now,
                "duration_sec": duration,
                "total_cost": sum(self.current_signal_prices.values()),
                "prices": self.current_signal_prices,
                "pnl": 1 - sum(self.current_signal_prices.values())
            })
            self.current_signal_start = None

    def heartbeat(self, ws):
        while True:
            time.sleep(20)
            try:
                ws.send("PING")
            except:
                break

    def on_error(self, ws, error):
        print(f"❌ WS Error: {error}")

    def on_close(self, ws, code, msg):
        print(f"🔌 Закрыто: {code} {msg}")
        self.close_current_signal(datetime.utcnow())
        self.reconnect_count += 1
        if self.reconnect_count > self.MAX_RECONNECTS:
            print("⚠️ Превышено число попыток reconnect, выходим")
            self.save_to_csv()
            return
        print("♻️ Переподключение через 5 секунд...")
        time.sleep(5)
        self.run_forever()

    def run_forever(self):
        self.ws = WebSocketApp(self.url,
                               on_open=self.on_open,
                               on_message=self.on_message,
                               on_error=self.on_error,
                               on_close=self.on_close)
        self.ws.run_forever()

    def save_to_csv(self):
        # Закрываем текущий сигнал, если он активен
        self.close_current_signal(datetime.utcnow())

        pd.DataFrame(self.events).to_csv("polymarket_events.csv", index=False)
        pd.DataFrame(self.signals).to_csv("polymarket_signals.csv", index=False)
        stats = {
            "total_signals": len(self.signals),
            "total_pnl": sum(s["pnl"] for s in self.signals),
            "avg_pnl_per_signal": sum(s["pnl"] for s in self.signals)/len(self.signals) if self.signals else 0,
            "avg_duration_sec": sum(s["duration_sec"] for s in self.signals)/len(self.signals) if self.signals else 0
        }
        pd.DataFrame([stats]).to_csv("polymarket_stats.csv", index=False)
        print("✅ Данные и статистика сохранены в CSV")

# ----------------------------
# 3️⃣ Запуск
# ----------------------------
ws_client = NegriskArbWS(token_ids)
try:
    ws_client.run_forever()
except KeyboardInterrupt:
    ws_client.save_to_csv()

✅ НАШЁЛ: Will Paris Saint-Germain FC win on 2026-04-08?...
💰 Volume: $8,786,656.03516992
🚀 Подписка на 3 токенов рынка
✅ Подключено к Polymarket WS
📤 Отправлена подписка на 3 токенов
❌ WS Error: Connection to remote host was lost.
🔌 Закрыто: None None
♻️ Переподключение через 5 секунд...


C:\Users\wertu\AppData\Local\Temp\ipykernel_4612\1738425140.py:137: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.close_current_signal(datetime.utcnow())


✅ Подключено к Polymarket WS
📤 Отправлена подписка на 3 токенов
❌ WS Error: Connection to remote host was lost.
🔌 Закрыто: None None
♻️ Переподключение через 5 секунд...


In [4]:
import requests
import json
import threading
import time
from websocket import WebSocketApp
from datetime import datetime
import pandas as pd

# ----------------------------
# 1️⃣ Выбор ликвидного negRisk рынка
# ----------------------------
url = 'https://gamma-api.polymarket.com/markets?active=true&limit=50&order=volume24hr&ascending=false'
markets = requests.get(url).json()

target_market = None
token_ids = []

for m in markets:
    tokens = m.get('clobTokenIds', [])
    if tokens and m.get('negRisk') and m.get('acceptingOrders'):
        print(f"✅ НАШЁЛ: {m['question']}...")
        print(f"💰 Volume: ${m['volume24hr']:,}")
        token_ids = tokens[:3]  # multi-outcome рынок
        target_market = m
        break

if not target_market:
    print("❌ Нет подходящих рынков")
    exit()

print(f"🚀 Подписка на {len(token_ids)} токенов рынка")

# ----------------------------
# 2️⃣ WS + Симуляция negrisk-арбитража
# ----------------------------
class NegriskArbSimulator:
    FEE = 0.001  # комиссия

    def __init__(self, asset_ids):
        self.asset_ids = asset_ids
        self.url = "wss://ws-subscriptions-clob.polymarket.com/ws/market"

        self.prices = {aid: 0.0 for aid in asset_ids}  # best ask
        self.best_bids = {aid: 0.0 for aid in asset_ids}

        self.position_open = False
        self.entry_time = None
        self.entry_cost = None
        self.entry_prices = None

        self.trades = []
        self.events = []

    # ----------------------------
    # WS callbacks
    # ----------------------------
    def heartbeat(self, ws):
        while True:
            time.sleep(10)
            try:
                ws.send("PING")
            except:
                break

    def on_open(self, ws):
        print("✅ WS подключен")
        sub = {
            "type": "market",
            "assets_ids": list(self.asset_ids),  # должно быть списком
            "custom_feature_enabled": True
        }
        ws.send(json.dumps(sub))
        print(f"📤 Подписка отправлена")

        threading.Thread(target=self.heartbeat, args=(ws,), daemon=True).start()

    def on_message(self, ws, message):
        if message.strip() == "PONG":
            print("📩 got PONG")
            return

        try:
            data = json.loads(message)
        except:
            return

        if isinstance(data, list):
            for item in data:
                self.process_event(item)
        else:
            self.process_event(data)

    def process_event(self, data):
        event_type = data.get("event_type")
        asset_id = data.get("asset_id")

        now = datetime.utcnow()

        # ----------------------------
        # Логируем все события
        # ----------------------------
        if event_type == "book":
            asks = data.get("asks", [])
            bids = data.get("bids", [])
            best_ask = float(asks[0][0]) if asks else 0
            best_bid = float(bids[0][0]) if bids else 0
            self.prices[asset_id] = best_ask
            self.best_bids[asset_id] = best_bid

            self.events.append({
                "timestamp": now,
                "asset_id": asset_id,
                "best_ask": best_ask,
                "best_bid": best_bid,
                "total_cost": sum(self.prices.values())
            })

            self.check_arbitrage(now)

        elif event_type == "last_trade_price":
            price = float(data.get("price", 0))
            size = float(data.get("size", 0))
            side = data.get("side", "?")
            self.events.append({
                "timestamp": now,
                "asset_id": asset_id,
                "price": price,
                "size": size,
                "side": side
            })

    # ----------------------------
    # Логика negrisk-арбитража
    # ----------------------------
    def check_arbitrage(self, now):
        # ждем, пока у всех токенов будут цены
        if 0 in self.prices.values():
            return

        total_cost = sum(self.prices.values())

        # ENTRY сигнал
        if (not self.position_open) and (0 < total_cost < 1):
            self.position_open = True
            self.entry_time = now
            self.entry_cost = total_cost
            self.entry_prices = self.prices.copy()

        # EXIT сигнал
        elif self.position_open and total_cost >= 1:
            self.close_position(now)

    def close_position(self, now):
        duration = (now - self.entry_time).total_seconds()
        pnl = (1 - self.entry_cost) * (1 - self.FEE)

        self.trades.append({
            "entry_time": self.entry_time,
            "exit_time": now,
            "duration_sec": duration,
            "entry_cost": self.entry_cost,
            "pnl": pnl,
            "entry_prices": self.entry_prices,
            "exit_prices": self.prices.copy()
        })

        self.position_open = False
        self.entry_time = None
        self.entry_cost = None
        self.entry_prices = None

    def on_close(self, ws, *args):
        print("🔌 WS закрыт")
        self.save_results()

    # ----------------------------
    # Запуск
    # ----------------------------
    def run(self):
        while True:
            try:
                ws = WebSocketApp(
                    self.url,
                    on_open=self.on_open,
                    on_message=self.on_message,
                    on_close=self.on_close
                )
                ws.run_forever()
            except Exception as e:
                print("Ошибка WS:", e)
            print("♻️ Переподключение через 5 сек...")
            time.sleep(5)

    # ----------------------------
    # Сохранение данных и статистики
    # ----------------------------
    def save_results(self):
        # Все RAW события
        if self.events:
            pd.DataFrame(self.events).to_csv("events.csv", index=False)

        # Сделки и PnL
        if self.trades:
            df = pd.DataFrame(self.trades)
            df.to_csv("trades.csv", index=False)

            stats = {
                "total_trades": len(df),
                "total_pnl": df["pnl"].sum(),
                "avg_pnl": df["pnl"].mean(),
                "avg_duration_sec": df["duration_sec"].mean(),
                "max_duration_sec": df["duration_sec"].max()
            }
            pd.DataFrame([stats]).to_csv("stats.csv", index=False)
            print("\n📊 РЕЗУЛЬТАТЫ:")
            for k, v in stats.items():
                print(f"{k}: {v}")
        else:
            print("Нет сделок для сохранения")

# ----------------------------
# 3️⃣ Запуск симулятора
# ----------------------------
sim = NegriskArbSimulator(token_ids)
try:
    sim.run()
except KeyboardInterrupt:
    sim.save_results()

✅ НАШЁЛ: Will Paris Saint-Germain FC win on 2026-04-08?...
💰 Volume: $8,786,656.03516992
🚀 Подписка на 3 токенов рынка
✅ WS подключен
📤 Подписка отправлена
📩 got PONG
📩 got PONG
📩 got PONG
📩 got PONG
📩 got PONG
📩 got PONG
📩 got PONG
📩 got PONG
📩 got PONG
📩 got PONG
📩 got PONG
📩 got PONG
📩 got PONG
📩 got PONG
📩 got PONG
📩 got PONG
📩 got PONG
📩 got PONG
📩 got PONG
📩 got PONG
📩 got PONG
📩 got PONG
📩 got PONG
📩 got PONG
📩 got PONG
📩 got PONG
📩 got PONG


C:\Users\wertu\AppData\Local\Temp\ipykernel_4612\90378824.py:97: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow()


📩 got PONG
📩 got PONG
📩 got PONG
📩 got PONG
📩 got PONG
📩 got PONG
📩 got PONG
📩 got PONG
📩 got PONG
📩 got PONG
📩 got PONG
📩 got PONG
📩 got PONG
📩 got PONG
📩 got PONG
🔌 WS закрыт
Нет сделок для сохранения
♻️ Переподключение через 5 сек...
Нет сделок для сохранения


In [1]:
import requests
import json
import threading
import time
from websocket import WebSocketApp
from datetime import datetime

# ----------------------------
# 1️⃣ Выбор ликвидного рынка (БЕЗ negRisk)
# ----------------------------
url = 'https://gamma-api.polymarket.com/markets?active=true&limit=50&order=volume24hr&ascending=false'
markets = requests.get(url).json()

target_market = None
token_ids = []

for m in markets:
    tokens = m.get('clobTokenIds', [])

    # ❌ УБРАЛИ: m.get('negRisk')
    # ❌ оставили только ликвидность + активность
    if tokens and m.get('acceptingOrders'):
        print(f"✅ НАШЁЛ: {m['question']}...")
        print(f"💰 Volume: ${m['volume24hr']:,}")

        token_ids = tokens[:3]  # multi-outcome рынок
        target_market = m
        break

if not target_market:
    print("❌ Нет подходящих рынков")
    exit()

print(f"🚀 Подписка на {len(token_ids)} токенов рынка")


# ----------------------------
# 2️⃣ WS + ПАРСИНГ (БЕЗ АРБИТРАЖА)
# ----------------------------
class MarketParser:

    def __init__(self, asset_ids):
        self.asset_ids = asset_ids
        self.url = "wss://ws-subscriptions-clob.polymarket.com/ws/market"

        self.prices = {aid: 0.0 for aid in asset_ids}
        self.best_bids = {aid: 0.0 for aid in asset_ids}

        self.events = []

    # ----------------------------
    # heartbeat
    # ----------------------------
    def heartbeat(self, ws):
        while True:
            time.sleep(10)
            try:
                ws.send("PING")
            except:
                break

    # ----------------------------
    # WS callbacks
    # ----------------------------
    def on_open(self, ws):
        print("✅ WS подключен")

        sub = {
            "type": "market",
            "assets_ids": list(self.asset_ids),
            "custom_feature_enabled": True
        }

        ws.send(json.dumps(sub))
        print("📤 Подписка отправлена")

        threading.Thread(target=self.heartbeat, args=(ws,), daemon=True).start()

    def on_message(self, ws, message):
        if message.strip() == "PONG":
            return

        try:
            data = json.loads(message)
        except:
            return

        if isinstance(data, list):
            for item in data:
                self.process_event(item)
        else:
            self.process_event(data)

    # ----------------------------
    # ПАРСИНГ СОБЫТИЙ (БЕЗ СТРАТЕГИИ)
    # ----------------------------
    def process_event(self, data):
        event_type = data.get("event_type")
        asset_id = data.get("asset_id")
        now = datetime.utcnow()

        # BOOK
        if event_type == "book":
            asks = data.get("asks", [])
            bids = data.get("bids", [])

            best_ask = float(asks[0][0]) if asks else 0
            best_bid = float(bids[0][0]) if bids else 0

            self.prices[asset_id] = best_ask
            self.best_bids[asset_id] = best_bid

            event = {
                "type": "book",
                "timestamp": now,
                "asset_id": asset_id,
                "best_ask": best_ask,
                "best_bid": best_bid
            }

            self.events.append(event)
            print(event)

        # LAST TRADE
        elif event_type == "last_trade_price":
            event = {
                "type": "trade",
                "timestamp": now,
                "asset_id": asset_id,
                "price": float(data.get("price", 0)),
                "size": float(data.get("size", 0)),
                "side": data.get("side")
            }

            self.events.append(event)
            print(event)

    # ----------------------------
    # RUN
    # ----------------------------
    def run(self):
        while True:
            try:
                ws = WebSocketApp(
                    self.url,
                    on_open=self.on_open,
                    on_message=self.on_message
                )
                ws.run_forever()

            except Exception as e:
                print("Ошибка WS:", e)

            print("♻️ Переподключение через 5 сек...")
            time.sleep(5)


# ----------------------------
# START
# ----------------------------
sim = MarketParser(token_ids)

try:
    sim.run()
except KeyboardInterrupt:
    print("🛑 остановлено")

✅ НАШЁЛ: US x Iran ceasefire by April 7?...
💰 Volume: $60,278,584.74312701
🚀 Подписка на 3 токенов рынка
✅ WS подключен
📤 Подписка отправлена


C:\Users\wertu\AppData\Local\Temp\ipykernel_10544\236269515.py:100: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow()


♻️ Переподключение через 5 сек...
✅ WS подключен
📤 Подписка отправлена
♻️ Переподключение через 5 сек...
🛑 остановлено


In [ ]:
import requests
import json
import threading
import time
from websocket import WebSocketApp
from datetime import datetime, timezone


# ----------------------------
# 1️⃣ Выбор ликвидного рынка
# ----------------------------
url = 'https://gamma-api.polymarket.com/markets?active=true&limit=50&order=volume24hr&ascending=false'
markets = requests.get(url).json()

target_market = None
token_ids = []

for m in markets:
    tokens = m.get('clobTokenIds', [])

    # ❌ negRisk убран
    if tokens and m.get('acceptingOrders'):
        print(f"✅ НАШЁЛ: {m['question']}...")
        print(f"💰 Volume: ${m['volume24hr']:,}")

        token_ids = tokens[:3]
        target_market = m
        break

if not target_market:
    print("❌ Нет подходящих рынков")
    exit()

print(f"🚀 Подписка на {len(token_ids)} токенов")


# ----------------------------
# 2️⃣ WS PARSER
# ----------------------------
class MarketParser:

    def __init__(self, asset_ids):
        self.asset_ids = asset_ids
        self.url = "wss://ws-subscriptions-clob.polymarket.com/ws/market"

        self.prices = {aid: 0.0 for aid in asset_ids}
        self.best_bids = {aid: 0.0 for aid in asset_ids}

        self.events = []

    def heartbeat(self, ws):
        while True:
            time.sleep(10)
            try:
                ws.send("PING")
            except:
                break

    def on_open(self, ws):
        print("✅ WS подключен")

        sub = {
            "type": "market",
            "assets_ids": list(self.asset_ids),
            "custom_feature_enabled": True
        }

        ws.send(json.dumps(sub))
        print("📤 Подписка отправлена")

        threading.Thread(target=self.heartbeat, args=(ws,), daemon=True).start()

    def on_message(self, ws, message):
        if message.strip() == "PONG":
            return

        try:
            data = json.loads(message)
        except:
            return

        if isinstance(data, list):
            for item in data:
                self.process_event(item)
        else:
            self.process_event(data)

    def process_event(self, data):
        event_type = data.get("event_type")
        asset_id = data.get("asset_id")

        # ✅ FIXED (правильный UTC)
        now = datetime.now(timezone.utc)

        if event_type == "book":
            asks = data.get("asks", [])
            bids = data.get("bids", [])

            best_ask = float(asks[0][0]) if asks else 0
            best_bid = float(bids[0][0]) if bids else 0

            self.prices[asset_id] = best_ask
            self.best_bids[asset_id] = best_bid

            event = {
                "type": "book",
                "timestamp": now,
                "asset_id": asset_id,
                "best_ask": best_ask,
                "best_bid": best_bid
            }

            self.events.append(event)
            print(event)

        elif event_type == "last_trade_price":
            event = {
                "type": "trade",
                "timestamp": now,
                "asset_id": asset_id,
                "price": float(data.get("price", 0)),
                "size": float(data.get("size", 0)),
                "side": data.get("side")
            }

            self.events.append(event)
            print(event)

    def run(self):
        while True:
            try:
                ws = WebSocketApp(
                    self.url,
                    on_open=self.on_open,
                    on_message=self.on_message
                )
                ws.run_forever()

            except Exception as e:
                print("Ошибка WS:", e)

            print("♻️ Переподключение через 5 сек...")
            time.sleep(5)


# ----------------------------
# START
# ----------------------------
sim = MarketParser(token_ids)

try:
    sim.run()
except KeyboardInterrupt:
    print("🛑 остановлено")

In [3]:
import requests
import json
import threading
import time
from websocket import WebSocketApp
from datetime import datetime, timezone

#Выбор ликвидного рынка

url = 'https://gamma-api.polymarket.com/markets?active=true&limit=50&order=volume24hr&ascending=false'
markets = requests.get(url).json()

target_market = None
token_ids = []

for m in markets:
    tokens = m.get('clobTokenIds', [])
    if tokens and m.get('acceptingOrders'):
        print(f"РЫНОК: {m['question']}...")
        print(f" Volume: ${m['volume24hr']:,}")

        token_ids = tokens[:3]
        target_market = m
        break

if not target_market:
    print(" Нет подходящих рынков")
    exit()

print(f" Подписка на {len(token_ids)} токенов")

#парсим

class MarketParser:

    def __init__(self, asset_ids):
        self.asset_ids = asset_ids
        self.url = "wss://ws-subscriptions-clob.polymarket.com/ws/market"

        self.prices = {aid: 0.0 for aid in asset_ids}
        self.best_bids = {aid: 0.0 for aid in asset_ids}

        self.events = []

    def heartbeat(self, ws):
        while True:
            time.sleep(10)
            try:
                ws.send("PING")
            except:
                break

    def on_open(self, ws):
        print(" WS подключен")

        sub = {
            "type": "market",
            "assets_ids": list(self.asset_ids),
            "custom_feature_enabled": True
        }

        ws.send(json.dumps(sub))
        print(" Подписка отправлена")

        threading.Thread(target=self.heartbeat, args=(ws,), daemon=True).start()

    def on_message(self, ws, message):

       
        if message.strip() == "PONG":
            print("got PONG")
            return

        try:
            data = json.loads(message)
        except:
            return

        if isinstance(data, list):
            for item in data:
                self.process_event(item)
        else:
            self.process_event(data)


    def process_event(self, data):
        event_type = data.get("event_type")
        asset_id = data.get("asset_id")

        now = datetime.now(timezone.utc)

        
        if event_type == "book":
            asks = data.get("asks", [])
            bids = data.get("bids", [])

            best_ask = float(asks[0][0]) if asks else 0
            best_bid = float(bids[0][0]) if bids else 0

            self.prices[asset_id] = best_ask
            self.best_bids[asset_id] = best_bid

            event = {
                "type": "book",
                "timestamp": now,
                "asset_id": asset_id,
                "best_ask": best_ask,
                "best_bid": best_bid
            }

            self.events.append(event)
            print(event)

        
        elif event_type == "last_trade_price":
            event = {
                "type": "trade",
                "timestamp": now,
                "asset_id": asset_id,
                "price": float(data.get("price", 0)),
                "size": float(data.get("size", 0)),
                "side": data.get("side")
            }

            self.events.append(event)
            print(event)


    def run(self):
        while True:
            try:
                ws = WebSocketApp(
                    self.url,
                    on_open=self.on_open,
                    on_message=self.on_message
                )
                ws.run_forever()

            except Exception as e:
                print("Ошибка WS:", e)

            print(" Переподключение через 5 сек...")
            time.sleep(5)


sim = MarketParser(token_ids)

try:
    sim.run()
except KeyboardInterrupt:
    print(" остановлено")

РЫНОК: US x Iran ceasefire by April 7?...
 Volume: $59,746,993.148844026
 Подписка на 3 токенов
 WS подключен
📤 Подписка отправлена
📩 got PONG
📩 got PONG
📩 got PONG
📩 got PONG
 Переподключение через 5 сек...
 остановлено
